## 01b - Anonymization

Pseudonymises the raw reviews before any further processing, so that no downstream
notebook, dataset or dashboard handles direct identifiers.

**Input**  : `data/raw/trenord_reviews_raw.csv` (read only, never modified)
**Output** : `data/processed/trenord_reviews_anon.csv`

Steps:
1. `review_id` → keyed hash (HMAC-SHA256 with `PSEUDO_KEY` from `.env`, first 12 hex characters).
   The mapping is stable across runs but cannot be reversed without the key.
2. `text` → ticket-like codes (8+ characters mixing letters and digits, or 8+ digits) → `[CODE]`.
3. `author` column dropped.

`reply_text` (the developer's public reply) is kept exactly as in the raw file: the checks
cell verifies that it is identical to the raw value for every row.

Only counts are printed: no review texts, names or keys appear in the outputs.

In [1]:
# --- IMPORTS ---
import hashlib
import hmac
import os
import re

import pandas as pd
from dotenv import find_dotenv, load_dotenv

# --- LOAD SECRET KEY ---
# The key is stored in .env at the repo root (gitignored); .env.example documents the variable.
# find_dotenv(usecwd=True) searches upward from the notebook folder.
load_dotenv(find_dotenv(usecwd=True))
PSEUDO_KEY = os.getenv("PSEUDO_KEY")

if PSEUDO_KEY is None or PSEUDO_KEY.strip() == "" or PSEUDO_KEY == "change_me":
    raise RuntimeError("PSEUDO_KEY not found: create .env from .env.example with a real key.")

# --- LOAD RAW DATA ---
# The raw file is only read; all changes are applied to a copy.
raw_df = pd.read_csv("../data/raw/trenord_reviews_raw.csv")
df = raw_df.copy()

print(f"Loaded {len(df):,} reviews, {df.shape[1]} columns")

Loaded 7,329 reviews, 10 columns


In [2]:
# --- STEP 1: PSEUDONYMISE review_id ---
# HMAC-SHA256 is a keyed hash: the same id always maps to the same value,
# but without the key the original id cannot be recovered or re-computed.
# 12 hex characters (48 bits) are ample to keep ~7,000 ids unique.

def pseudonymize_id(original_id):
    key_bytes = PSEUDO_KEY.encode("utf-8")
    message_bytes = str(original_id).encode("utf-8")
    digest = hmac.new(key_bytes, message_bytes, hashlib.sha256).hexdigest()
    return digest[:12]

df["review_id"] = df["review_id"].apply(pseudonymize_id)

print(f"Pseudonymised ids: {df['review_id'].nunique():,} unique out of {len(df):,}")

Pseudonymised ids: 7,329 unique out of 7,329


In [3]:
# --- STEP 2: MASK CODES IN REVIEW TEXT ---
# Long alphanumeric strings are typically ticket, order or card numbers.
# Mixed letters+digits (8+ chars) are masked first, then pure digit runs (8+ digits).

MIXED_CODE_PATTERN = re.compile(r"\b(?=[A-Za-z0-9]*\d)(?=[A-Za-z0-9]*[A-Za-z])[A-Za-z0-9]{8,}\b")
DIGIT_CODE_PATTERN = re.compile(r"(?<!\d)\d{8,}(?!\d)")

masked_texts = []
code_replacements = 0
for text in df["text"]:
    if not isinstance(text, str):
        masked_texts.append(text)
        continue
    text, mixed_count = MIXED_CODE_PATTERN.subn("[CODE]", text)
    text, digit_count = DIGIT_CODE_PATTERN.subn("[CODE]", text)
    code_replacements += mixed_count + digit_count
    masked_texts.append(text)
df["text"] = masked_texts

print(f"[CODE] replacements in text: {code_replacements}")

[CODE] replacements in text: 5


In [4]:
# --- STEP 3: DROP author ---
# Reviewer names are direct identifiers and are not needed downstream.
df = df.drop(columns=["author"])

print(f"Columns kept: {df.columns.tolist()}")

Columns kept: ['review_id', 'rating', 'text', 'thumbs_up', 'app_version', 'date', 'has_reply', 'reply_text', 'reply_date']


In [5]:
# --- CHECKS (counts only) ---
original_ids = set(raw_df["review_id"].astype(str))
pseudonymised_ids = set(df["review_id"].astype(str))

# reply_text must be untouched: compared row by row with the raw file (same row order).
# Two missing values count as identical.
replies_different = 0
for raw_reply, anon_reply in zip(raw_df["reply_text"], df["reply_text"]):
    both_missing = pd.isna(raw_reply) and pd.isna(anon_reply)
    if not both_missing and raw_reply != anon_reply:
        replies_different += 1

checks = {
    "same row count as raw": len(df) == len(raw_df),
    "author column absent": "author" not in df.columns,
    "review_id unique": df["review_id"].is_unique,
    "no original review_id present": len(original_ids & pseudonymised_ids) == 0,
    "reply_text identical to raw for every row": replies_different == 0,
}

print(f"Rows raw / anon                       : {len(raw_df):,} / {len(df):,}")
print(f"Replies (non-null) raw / anon         : {raw_df['reply_text'].notna().sum():,} / {df['reply_text'].notna().sum():,}")
print(f"Replies different from raw            : {replies_different}")
print(f"[CODE] replacements                   : {code_replacements}")
print()
for check_name, passed in checks.items():
    print(f"{'PASS' if passed else 'FAIL'}  {check_name}")

if not all(checks.values()):
    raise AssertionError("Anonymisation checks failed — output not saved.")

Rows raw / anon                       : 7,329 / 7,329
Replies (non-null) raw / anon         : 3,052 / 3,052
Replies different from raw            : 0
[CODE] replacements                   : 5

PASS  same row count as raw
PASS  author column absent
PASS  review_id unique
PASS  no original review_id present
PASS  reply_text identical to raw for every row


In [6]:
# --- SAVE ANONYMISED DATA ---
df.to_csv("../data/processed/trenord_reviews_anon.csv", index=False, encoding="utf-8-sig")

print(f"Saved {len(df):,} reviews to ../data/processed/trenord_reviews_anon.csv")

Saved 7,329 reviews to ../data/processed/trenord_reviews_anon.csv
